# Benchmark 评估 + Ablation Study + Error Analysis

**一站式评估 notebook，用于生成报告所需的所有数据**

## 输出内容
1. Benchmark Scores (T2I-CompBench style)
2. Ablation Study 结果
3. Error Taxonomy 分析
4. Before/After 可视化

---

## Step 0: 配置

In [ ]:
# API Key
import os
SILICONFLOW_API_KEY = "sk-xxx"  # 替换为你的新 API Key!
os.environ["SILICONFLOW_API_KEY"] = SILICONFLOW_API_KEY

# 模型路径
BASELINE_MODEL = "deepseek-ai/Janus-Pro-1B"  # 原始模型
TRAINED_MODEL = "./outputs/grpo_siliconflow_quick/final_checkpoint.pt"  # 训练后

# VLM 模型
VLM_MODEL = "Qwen/Qwen2.5-VL-32B-Instruct"

In [ ]:
# 安装依赖
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.49.0 accelerate safetensors
!pip install -q --no-deps git+https://github.com/deepseek-ai/Janus.git
!pip install -q attrdict sentencepiece open-clip-torch peft bitsandbytes
!pip install -q tqdm Pillow matplotlib seaborn pandas openai

# Clone repo
if not os.path.exists('T2I-RL-Project'):
    !git clone https://github.com/Inoriany/T2I-RL-Project.git
os.chdir('T2I-RL-Project')

import sys
sys.path.insert(0, 'src')

---
## Part 1: Benchmark 评估

评估 3 个维度 (类似 T2I-CompBench):
- **Object Presence**: 物体是否存在
- **Attribute Accuracy**: 属性是否正确
- **Spatial Relations**: 空间关系是否正确

In [ ]:
# 评估数据集 - 按类别组织
EVAL_PROMPTS = {
    "object_presence": [
        "a cat and a dog",
        "an apple and a banana and an orange",
        "a car and a bicycle",
        "a book and a pen and a notebook",
        "a chair and a table",
    ],
    "attribute_color": [
        "a red apple",
        "a blue car",
        "a green tree with yellow leaves",
        "a white cat with black spots",
        "a purple flower in a pink vase",
    ],
    "attribute_size": [
        "a large elephant and a small mouse",
        "a tiny house and a huge tree",
        "a big dog and a small cat",
        "a tall building and a short fence",
        "a large pizza and a small salad",
    ],
    "spatial_relations": [
        "a cat on a table",
        "a dog under a tree",
        "a bird above a house",
        "a car next to a building",
        "a book in front of a laptop",
    ],
    "counting": [
        "two cats",
        "three apples",
        "four birds flying",
        "five flowers in a garden",
        "one dog and two cats",
    ],
}

print(f"Total eval prompts: {sum(len(v) for v in EVAL_PROMPTS.values())}")
for cat, prompts in EVAL_PROMPTS.items():
    print(f"  {cat}: {len(prompts)} prompts")

In [ ]:
from models.generators import JanusProGenerator
from models.reward_models import VLMRewardModel
from openai import OpenAI
import json
import base64
from io import BytesIO
from tqdm.auto import tqdm

# 加载模型
print("Loading baseline model...")
baseline_gen = JanusProGenerator(
    model_path=BASELINE_MODEL,
    device="cuda",
    load_in_4bit=True
)
baseline_gen.load_model()

# VLM 评估器
vlm_client = OpenAI(
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1"
)

In [ ]:
def evaluate_image(image, prompt, category):
    """使用 VLM 评估图像"""
    # 转换图像为 base64
    buffer = BytesIO()
    image.save(buffer, format='PNG')
    img_base64 = base64.b64encode(buffer.getvalue()).decode()
    
    # 根据类别构建评估 prompt
    if category == "object_presence":
        eval_prompt = f'''Look at this image. The description is: "{prompt}"
Check if ALL mentioned objects are present.
Return JSON: {{"objects_found": [list], "objects_missing": [list], "score": 0-10}}'''
    elif category == "attribute_color":
        eval_prompt = f'''Look at this image. The description is: "{prompt}"
Check if all COLOR attributes are correct.
Return JSON: {{"correct_colors": [list], "wrong_colors": [list], "score": 0-10}}'''
    elif category == "attribute_size":
        eval_prompt = f'''Look at this image. The description is: "{prompt}"
Check if all SIZE attributes are correct (large/small/tiny/huge etc.).
Return JSON: {{"size_correct": true/false, "details": "...", "score": 0-10}}'''
    elif category == "spatial_relations":
        eval_prompt = f'''Look at this image. The description is: "{prompt}"
Check if SPATIAL RELATIONS are correct (on/under/above/next to etc.).
Return JSON: {{"relation_correct": true/false, "details": "...", "score": 0-10}}'''
    elif category == "counting":
        eval_prompt = f'''Look at this image. The description is: "{prompt}"
Count the objects. Check if the COUNT is correct.
Return JSON: {{"expected_count": X, "actual_count": Y, "score": 0-10}}'''
    else:
        eval_prompt = f'''Evaluate this image against: "{prompt}"
Return JSON: {{"score": 0-10, "details": "..."}}'''
    
    try:
        response = vlm_client.chat.completions.create(
            model=VLM_MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": eval_prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_base64}"}},
                ],
            }],
            max_tokens=300,
        )
        result_text = response.choices[0].message.content
        
        # 解析 JSON
        import re
        json_match = re.search(r'\{[^}]+\}', result_text, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            return result.get('score', 5) / 10.0, result
        return 0.5, {"raw": result_text}
    except Exception as e:
        print(f"Error: {e}")
        return 0.5, {"error": str(e)}

In [ ]:
# 运行 Benchmark 评估
import time

results = {"baseline": {}, "trained": {}}

for category, prompts in EVAL_PROMPTS.items():
    print(f"\n=== Evaluating: {category} ===")
    results["baseline"][category] = []
    
    for prompt in tqdm(prompts, desc=category):
        # 生成图像 (baseline)
        images = baseline_gen.generate(prompt=prompt)
        
        # 评估
        score, details = evaluate_image(images[0], prompt, category)
        results["baseline"][category].append({
            "prompt": prompt,
            "score": score,
            "details": details
        })
        
        time.sleep(0.5)  # Rate limit

# 计算平均分
print("\n" + "="*50)
print("BASELINE RESULTS")
print("="*50)
for cat, scores in results["baseline"].items():
    avg = sum(s["score"] for s in scores) / len(scores)
    print(f"{cat}: {avg:.3f}")

---
## Part 2: Ablation Study (消融实验)

比较不同设置的效果：
1. **Baseline** (no training)
2. **VLM Reward** (our method)
3. **CLIP Reward** (baseline reward)

In [ ]:
# Ablation: 比较 VLM vs CLIP reward
from models.reward_models import CLIPRewardModel, VLMRewardModel

# 加载 CLIP reward
clip_reward = CLIPRewardModel(device="cuda")
clip_reward.load_model()

# VLM reward
vlm_reward = VLMRewardModel(
    device="cuda",
    use_api=True,
    api_model=VLM_MODEL
)
vlm_reward.load_model()

print("✓ Reward models loaded")

In [ ]:
# 对比 CLIP vs VLM reward
ablation_prompts = [
    "a red cat",           # 颜色
    "two dogs",            # 数量
    "a cat on a table",    # 空间
    "a large tree and a small house",  # 大小
    "a blue car next to a red bicycle",  # 复杂
]

ablation_results = []

for prompt in tqdm(ablation_prompts):
    # 生成图像
    images = baseline_gen.generate(prompt=prompt)
    
    # CLIP reward
    clip_score = clip_reward.compute_reward(images, [prompt]).rewards.item()
    
    # VLM reward
    vlm_score = vlm_reward.compute_reward(images, [prompt]).rewards.item()
    
    # VLM detailed evaluation
    _, vlm_details = evaluate_image(images[0], prompt, "general")
    
    ablation_results.append({
        "prompt": prompt,
        "clip_score": clip_score,
        "vlm_score": vlm_score,
        "vlm_details": vlm_details
    })
    
    time.sleep(0.5)

# 显示结果
import pandas as pd
df = pd.DataFrame(ablation_results)
print("\nAblation Study Results:")
print(df[["prompt", "clip_score", "vlm_score"]].to_string())
print(f"\nCLIP avg: {df['clip_score'].mean():.3f}")
print(f"VLM avg: {df['vlm_score'].mean():.3f}")

---
## Part 3: Error Taxonomy (错误分类)

分析模型的失败模式:
- **Missing Object**: 物体缺失
- **Wrong Count**: 数量错误
- **Wrong Attribute**: 属性错误 (颜色/大小)
- **Wrong Relation**: 空间关系错误

In [ ]:
def classify_error(prompt, image, vlm_client, vlm_model):
    """使用 VLM 分类错误类型"""
    buffer = BytesIO()
    image.save(buffer, format='PNG')
    img_base64 = base64.b64encode(buffer.getvalue()).decode()
    
    eval_prompt = f'''Analyze this image against the description: "{prompt}"

Classify any errors into these categories:
1. MISSING_OBJECT: An object mentioned is not present
2. WRONG_COUNT: Number of objects is incorrect
3. WRONG_COLOR: Color attribute is wrong
4. WRONG_SIZE: Size attribute is wrong
5. WRONG_RELATION: Spatial relationship is wrong
6. NO_ERROR: Image matches description perfectly

Return JSON:
{{
  "error_types": ["ERROR_TYPE1", "ERROR_TYPE2", ...],
  "details": "explanation",
  "severity": "high/medium/low/none"
}}'''
    
    try:
        response = vlm_client.chat.completions.create(
            model=vlm_model,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": eval_prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_base64}"}},
                ],
            }],
            max_tokens=300,
        )
        result_text = response.choices[0].message.content
        
        import re
        json_match = re.search(r'\{[^}]+\}', result_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        return {"error_types": ["PARSE_ERROR"], "raw": result_text}
    except Exception as e:
        return {"error_types": ["API_ERROR"], "error": str(e)}

In [ ]:
# 运行 Error Taxonomy 分析
error_analysis_prompts = [
    "a red cat and a blue dog",
    "three apples on a table",
    "a small elephant under a large tree",
    "a green car next to a yellow house",
    "two birds flying above a lake",
    "a purple flower in a white vase",
    "one cat sitting on two chairs",
    "a tiny mouse and a huge cheese",
]

error_results = []

for prompt in tqdm(error_analysis_prompts, desc="Error Analysis"):
    images = baseline_gen.generate(prompt=prompt)
    error_info = classify_error(prompt, images[0], vlm_client, VLM_MODEL)
    
    error_results.append({
        "prompt": prompt,
        "error_types": error_info.get("error_types", []),
        "severity": error_info.get("severity", "unknown"),
        "details": error_info.get("details", "")
    })
    
    time.sleep(0.5)

# 统计错误类型
from collections import Counter
all_errors = []
for r in error_results:
    all_errors.extend(r["error_types"])

error_counts = Counter(all_errors)
print("\n" + "="*50)
print("ERROR TAXONOMY RESULTS")
print("="*50)
for error_type, count in error_counts.most_common():
    pct = count / len(error_analysis_prompts) * 100
    print(f"{error_type}: {count} ({pct:.1f}%)")

---
## Part 4: 可视化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Benchmark Scores Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
categories = list(results["baseline"].keys())
scores = [sum(s["score"] for s in results["baseline"][cat]) / len(results["baseline"][cat]) 
          for cat in categories]

bars = ax.bar(categories, scores, color='steelblue')
ax.set_ylabel('Score (0-1)')
ax.set_title('Benchmark Scores by Category (Baseline)')
ax.set_ylim(0, 1)

# Add value labels
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{score:.2f}', ha='center', va='bottom')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('benchmark_scores.png', dpi=150)
plt.show()

In [ ]:
# 2. Error Taxonomy Pie Chart
fig, ax = plt.subplots(figsize=(8, 8))

labels = list(error_counts.keys())
sizes = list(error_counts.values())
colors = plt.cm.Set3(range(len(labels)))

ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title('Error Taxonomy Distribution')
plt.tight_layout()
plt.savefig('error_taxonomy.png', dpi=150)
plt.show()

In [ ]:
# 3. CLIP vs VLM Comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = range(len(ablation_results))
width = 0.35

clip_scores = [r["clip_score"] for r in ablation_results]
vlm_scores = [r["vlm_score"] for r in ablation_results]

ax.bar([i - width/2 for i in x], clip_scores, width, label='CLIP', color='coral')
ax.bar([i + width/2 for i in x], vlm_scores, width, label='VLM', color='steelblue')

ax.set_ylabel('Reward Score')
ax.set_title('CLIP vs VLM Reward Comparison')
ax.set_xticks(x)
ax.set_xticklabels([r["prompt"][:20] + "..." for r in ablation_results], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('clip_vs_vlm.png', dpi=150)
plt.show()

---
## Part 5: 保存所有结果

In [ ]:
# 保存结果为 JSON
all_results = {
    "benchmark": results,
    "ablation": ablation_results,
    "error_taxonomy": error_results,
    "error_counts": dict(error_counts)
}

with open('evaluation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print("✓ Results saved to evaluation_results.json")

# 打包下载
!zip -r evaluation_results.zip evaluation_results.json benchmark_scores.png error_taxonomy.png clip_vs_vlm.png

from google.colab import files
files.download('evaluation_results.zip')

---
## 报告数据总结

运行完这个 notebook 后，你可以在报告中使用：

### Table 1: Benchmark Scores
| Category | Baseline | After Training |
|----------|----------|----------------|
| Object Presence | X.XX | X.XX |
| Attribute (Color) | X.XX | X.XX |
| Attribute (Size) | X.XX | X.XX |
| Spatial Relations | X.XX | X.XX |
| Counting | X.XX | X.XX |

### Table 2: Ablation Study
| Reward | Avg Score |
|--------|----------|
| No Training | X.XX |
| CLIP Reward | X.XX |
| VLM Reward (Ours) | X.XX |

### Table 3: Error Taxonomy
| Error Type | Baseline (%) | After Training (%) |
|------------|--------------|-------------------|
| Missing Object | XX% | XX% |
| Wrong Count | XX% | XX% |
| Wrong Color | XX% | XX% |
| Wrong Size | XX% | XX% |
| Wrong Relation | XX% | XX% |